In [7]:
%%writefile distributed_hello_world.py
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp


def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

def distributed_demo(rank, world_size):
    setup(rank, world_size)
    data = torch.randint(0, 10, (3,))
    print(f"rank {rank} data (before all-reduce): {data}")
    dist.all_reduce(data,  async_op=False)
    print(f"rank {rank} data (after all-reduce): {data}")

if __name__ == "__main__":
    world_size = 6
    mp.spawn(distributed_demo, args=(world_size, ), nprocs=world_size, join=True)

Overwriting distributed_hello_world.py


In [8]:
! python distributed_hello_world.py

[Gloo] Rank 0 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
[Gloo] Rank 2 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
[Gloo] Rank 1 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
[Gloo] Rank 5 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
[Gloo] Rank 4 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
rank 1 data (before all-reduce): tensor([5, 7, 7])
[Gloo] Rank 3 is connected to 5 peer ranks. Expected number of connected peer ranks is : 5
rank 0 data (before all-reduce): tensor([4, 9, 8])
rank 5 data (before all-reduce): tensor([2, 5, 1])
rank 3 data (before all-reduce): tensor([9, 0, 2])
rank 2 data (before all-reduce): tensor([5, 6, 9])
rank 4 data (before all-reduce): tensor([0, 5, 7])
rank 5 data (after all-reduce): tensor([25, 32, 34])
rank 0 data (after all-reduce): tensor([25, 32, 34])
rank 4 data (after all-reduce): tensor([25